# Building MCP from scratch
## Our own server, client, and host — end to end

Last lesson we saw the **N × M problem**: every (LLM app, tool) pair needed a custom integration. The escape hatch was a standard protocol.

Today we **build** that protocol's pieces with our own hands — two small servers, a client that talks to them, and a host that wires them to an LLM (OpenAI Responses API). No agent libraries. No magic.

By the end we'll have a chat loop where a question like *"multiply the word count of this sentence by 7"* gets answered by the LLM coordinating two completely separate servers it has never seen before.

## The three roles in one picture

```
┌──────────────────────── HOST (our app) ────────────────────────┐
│                                                                │
│    ┌─────────┐              LLM  ←→  OpenAI Responses API      │
│    │ Client1 │ ─── stdio ───▶ 📦 Math Server                   │
│    ├─────────┤                                                 │
│    │ Client2 │ ─── stdio ───▶ 📝 Text Server                   │
│    └─────────┘                                                 │
└────────────────────────────────────────────────────────────────┘
```

- **Server** — exposes tools. Knows nothing about LLMs, nothing about other servers.
- **Client** — speaks MCP to exactly one server. Discovers what's there, calls tools.
- **Host** — our application. Runs an LLM, runs N clients, and routes tool calls between them.

We'll build each role in this order: server → client → host.

In [27]:
# Install the bits we need
#!pip install -q mcp openai

In [28]:
import os, json
from dotenv import load_dotenv

import textwrap

import truststore
truststore.inject_into_ssl()

def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

API key loaded successfully.


---
## Step 1 — Build Server #1: the 🧮 Math server

Notice what this file contains and what it does *not*:

✅ It defines tools and runs an MCP server.
❌ It doesn't import `openai`. It doesn't know what an LLM is. It doesn't know any other server exists.

That independence is the point of the protocol.

In [29]:
%%writefile math_server.py
"""A tiny MCP server that exposes math tools over stdio."""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("math")

@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

@mcp.tool()
def power(base: float, exponent: float) -> float:
    """Raise base to the given exponent."""
    return base ** exponent

if __name__ == "__main__":
    # Default transport is stdio — the server reads JSON-RPC from stdin
    # and writes to stdout. Perfect for being spawned as a subprocess.
    mcp.run()


Overwriting math_server.py


## Step 2 — Build Server #2: the 📝 Text server

Again: zero knowledge of LLMs, zero knowledge of the math server. Different file, different process, different domain.

In [30]:
%%writefile text_server.py
"""A tiny MCP server that exposes text tools over stdio."""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("text")

@mcp.tool()
def uppercase(text: str) -> str:
    """Convert text to uppercase."""
    return text.upper()

@mcp.tool()
def word_count(text: str) -> int:
    """Count the number of whitespace-separated words in text."""
    return len(text.split())

@mcp.tool()
def reverse(text: str) -> str:
    """Return the text with its characters reversed."""
    return text[::-1]

if __name__ == "__main__":
    mcp.run()


Overwriting text_server.py


### 🔎 What we just did

We now have **two standalone MCP servers** as Python files on disk. Each one:

- Uses the `@mcp.tool()` decorator to register functions. FastMCP reads the Python signatures and docstrings and **auto-generates the JSON schemas** for us — no more hand-writing `{"type": "object", "properties": ...}`.
- Runs over **stdio** — anyone who can spawn a subprocess can talk to it.
- Could be published, shared, reused by any MCP-compatible host — including Claude Desktop, Cursor, or our own host below.

---
## Step 3 — Build a Client and talk to one server

A **client** is the thing that speaks MCP to a server. We'll:

1. Spawn `math_server.py` as a subprocess.
2. Open an MCP `ClientSession` over its stdin/stdout.
3. **Discover** its tools (we don't hardcode anything!).
4. **Call** one.

This is the protocol in its purest form — no LLM involved yet.

In [31]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Tell MCP how to launch the server: just run it with our Python interpreter.
math_params = StdioServerParameters(command=sys.executable, args=["math_server.py"])

async with stdio_client(math_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        # 1) DISCOVERY — ask the server what tools it has
        listed = await session.list_tools()
        print("Tools exposed by the math server:")
        for t in listed.tools:
            print(f"  • {t.name:10} — {t.description}")

        # 2) INVOCATION — call one
        result = await session.call_tool("multiply", {"a": 6, "b": 7})
        print(f"\nmultiply(6, 7) = {result.content[0].text}")


Tools exposed by the math server:
  • add        — Add two numbers.
  • multiply   — Multiply two numbers.
  • power      — Raise base to the given exponent.

multiply(6, 7) = 42.0


### 🔎 The magic moment

We did **not** import `add`, `multiply`, or `power`. We didn't import anything from `math_server.py` at all. We just:

1. Launched it as a subprocess.
2. Asked it *"what can you do?"* and it told us.
3. Asked it *"please multiply 6 by 7"* and it did.

That's **runtime discovery**. A host written today can use tools that didn't exist when it was written. This is the thing that function-calling-with-hardcoded-schemas could never give us.

---
## Step 4 — Build the Host

The host is the layer that:

1. Runs **multiple** MCP clients (one per server).
2. Aggregates all discovered tools into one pool.
3. Runs the LLM loop: send tools + message → if LLM asks for a tool, route it to the right client → send result back → repeat.

The key insight: **the host doesn't care what tools exist.** It just passes through whatever any connected server exposes.

In [32]:
import json
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


class MCPHost:
    """Manages multiple MCP client sessions and exposes their tools as one pool."""

    def __init__(self):
        self.session_by_tool = {}   # tool_name -> ClientSession
        self.openai_tools = []      # tool schemas in OpenAI Responses format
        self._stack = AsyncExitStack()

    async def add_server(self, command, args):
        """Spawn a server and register all its tools."""
        params = StdioServerParameters(command=command, args=args)
        read, write = await self._stack.enter_async_context(stdio_client(params))
        session = await self._stack.enter_async_context(ClientSession(read, write))
        await session.initialize()

        listed = await session.list_tools()
        for t in listed.tools:
            self.session_by_tool[t.name] = session
            # Translate MCP's tool shape -> OpenAI Responses tool shape.
            # MCP gives us: name, description, inputSchema (JSON schema).
            # OpenAI wants: type, name, description, parameters.
            self.openai_tools.append({
                "type": "function",
                "name": t.name,
                "description": t.description or "",
                "parameters": t.inputSchema,
            })
        print(f"  connected — tools: {[t.name for t in listed.tools]}")

    async def call(self, tool_name, args):
        """Route a tool call to the right server."""
        session = self.session_by_tool[tool_name]
        result = await session.call_tool(tool_name, args)
        # result.content is a list of content parts; grab their text
        return "\n".join(c.text for c in result.content if hasattr(c, "text"))

    async def close(self):
        await self._stack.aclose()


### Now connect to BOTH servers

In [33]:
host = MCPHost()

print("Connecting to math server…")
await host.add_server(sys.executable, ["math_server.py"])

print("Connecting to text server…")
await host.add_server(sys.executable, ["text_server.py"])

print(f"\nHost now knows {len(host.openai_tools)} tools from {len(set(host.session_by_tool.values()))} servers.")


Connecting to math server…
  connected — tools: ['add', 'multiply', 'power']
Connecting to text server…
  connected — tools: ['uppercase', 'word_count', 'reverse']

Host now knows 6 tools from 2 servers.


In [34]:
# Let's peek at what the unified tool list looks like — this is exactly
# what we will pass to OpenAI. Nothing hand-written.
from pprint import pprint
pprint(host.openai_tools)


[{'description': 'Add two numbers.',
  'name': 'add',
  'parameters': {'properties': {'a': {'title': 'A', 'type': 'number'},
                                'b': {'title': 'B', 'type': 'number'}},
                 'required': ['a', 'b'],
                 'title': 'addArguments',
                 'type': 'object'},
  'type': 'function'},
 {'description': 'Multiply two numbers.',
  'name': 'multiply',
  'parameters': {'properties': {'a': {'title': 'A', 'type': 'number'},
                                'b': {'title': 'B', 'type': 'number'}},
                 'required': ['a', 'b'],
                 'title': 'multiplyArguments',
                 'type': 'object'},
  'type': 'function'},
 {'description': 'Raise base to the given exponent.',
  'name': 'power',
  'parameters': {'properties': {'base': {'title': 'Base', 'type': 'number'},
                                'exponent': {'title': 'Exponent',
                                             'type': 'number'}},
                 'required

### 🔎 Look at that list again

Every schema there was **auto-generated** from Python type hints in the server files, then **auto-translated** to OpenAI's format by the host. We did not hand-write a single JSON schema today.

Compare that to last lesson's mountain of `parameters` / `input_schema` / `function_declarations` nightmares.

---
## Step 5 — Wire in the LLM (OpenAI Responses API)

The host's final job: send the discovered tools to the LLM, handle any tool calls the LLM makes, and loop until the LLM produces a final text answer.

> Set your `OPENAI_API_KEY` environment variable before running this cell.

In [35]:
import json
from openai import OpenAI

openai_client = OpenAI()
MODEL = "gpt-5-nano"


async def chat(host, user_message, verbose=True):
    """Run one user message through an MCP-backed tool-use loop."""
    input_items = [{"role": "user", "content": user_message}]

    while True:
        response = openai_client.responses.create(
            instructions="You must use the provided tools for all operations if possible. Do not compute anything yourself, even trivial values. Chain tools when needed.",
            model=MODEL,
            input=input_items,
            tools=host.openai_tools,
        )

        # Collect any tool calls the model emitted this turn
        tool_calls = [item for item in response.output if item.type == "function_call"]

        if not tool_calls:
            # No more tool calls — the model is done; return its text reply
            print("No more tool calls — the model is done; returning its text reply.")
            return response.output_text

        # For each tool call: execute via the host and append result
        for tc in tool_calls:
            args = json.loads(tc.arguments)
            if verbose:
                print(f"🔧 {tc.name}({args})")

            result = await host.call(tc.name, args)

            if verbose:
                print(f"   → {result}")

            # Echo the function_call back, then supply its output
            input_items.append({
                "type": "function_call",
                "call_id": tc.call_id,
                "name": tc.name,
                "arguments": tc.arguments,
            })
            input_items.append({
                "type": "function_call_output",
                "call_id": tc.call_id,
                "output": result,
            })


### Test 1 — a math-only question

In [36]:
answer = await chat(host, "What is 13 times 29, then raised to the power of 2?")
print(f"\n💬 {answer}")


🔧 multiply({'a': 13, 'b': 29})
   → 377.0
🔧 power({'base': 377, 'exponent': 2})
   → 142129.0
No more tool calls — the model is done; returning its text reply.

💬 13 × 29 = 377
377^2 = 142,129

Final result: 142,129


### Test 2 — a text-only question

In [37]:
answer = await chat(host, "Reverse the string 'model context protocol' and then shout it.")
print(f"\n💬 {answer}")


🔧 reverse({'text': 'model context protocol'})
   → locotorp txetnoc ledom
🔧 uppercase({'text': 'locotorp txetnoc ledom'})
   → LOCOTORP TXETNOC LEDOM
No more tool calls — the model is done; returning its text reply.

💬 LOCOTORP TXETNOC LEDOM!


### Test 3 — needs BOTH servers in one conversation

This is the payoff. The LLM has never seen our servers before, and yet it will cross the Math/Text boundary freely — because to it they're just tools in one pool.

In [38]:
answer = await chat(
    host,
    "Count the words in the sentence 'MCP makes tools discoverable at runtime' "
    "and then multiply that count by 13.5."
)
print(f"\n💬 {answer}")


🔧 word_count({'text': 'MCP makes tools discoverable at runtime'})
   → 6
🔧 multiply({'a': 6, 'b': 13.5})
   → 81.0
No more tool calls — the model is done; returning its text reply.

💬 The result is 81.0.


---
## What we achieved vs last lesson

| Before (N × M) | After (with MCP) |
|---|---|
| Hand-written JSON schema for every tool | Auto-generated from Python type hints |
| Dispatcher `if/elif` chain | `self.session_by_tool[name]` lookup |
| Rewrite the loop per LLM provider | One loop; swap the host's LLM call |
| Tools hardcoded into the app at build time | **Discovered at runtime** via `list_tools()` |
| Your coworker rebuilds everything | They point their host at our server files |

### What each piece owns

- **Server** — "here are my tools and their schemas." Nothing else.
- **Client** — "I speak MCP to one server." Reusable across hosts.
- **Host** — "I run an LLM and N clients, and I translate between them."

### The one translation we still did

Notice: the host still converts MCP's `inputSchema` → OpenAI's `parameters` shape. That's the **N** side of the old problem — LLM providers still have different tool-calling formats. But now that translation lives in *one place* (the host), not sprinkled across every tool integration.

Swap OpenAI for Anthropic? Rewrite only the `chat()` function and the shape of `openai_tools`. The servers, the clients, and the host's server-management code don't move.

**That's the N + M win, made concrete.**

---
## Bonus — The same three tests, via the OpenAI Agents SDK

We just built the host loop by hand — `openai_tools`, `function_call`, `function_call_output`, the while-loop, the dispatcher. Great for understanding, verbose for real code.

The **OpenAI Agents SDK** collapses most of that: you point it at MCP servers and it handles discovery, schema translation, the tool-call loop, and result feeding for you. Same servers, same tests — zero routing code on our side.

```
Before (our host):   [spawn] → [list_tools] → [translate schemas] → [while: call LLM / route tool / feed result] → answer
After (Agents SDK):  [spawn] → Runner.run(...) → answer
```

The servers and clients we built are untouched — that's the MCP payoff. Only the *host* shrinks.

> Install once if you don't have it: `pip install openai-agents`

In [ ]:
# ══════════════════════════════════════════════════════════════
# OpenAI Agents SDK + MCP — the cleanest approach
# Same math_server.py + text_server.py, all three tests, zero routing code.
# ══════════════════════════════════════════════════════════════
from agents import Agent, Runner
from agents.mcp import MCPServerStdio


async def agents_sdk_demo():
    # Point the Agents SDK at BOTH of our MCP servers.
    math_mcp = MCPServerStdio(
        params={"command": sys.executable, "args": ["math_server.py"]},
        cache_tools_list=True,
    )
    text_mcp = MCPServerStdio(
        params={"command": sys.executable, "args": ["text_server.py"]},
        cache_tools_list=True,
    )

    # The SDK manages the connection lifecycle for us.
    async with math_mcp, text_mcp:
        agent = Agent(
            name="MCP Demo Assistant",
            model=MODEL,
            instructions=(
                "You have access to math and text tools over MCP. "
                "Use them for every operation — do not compute anything yourself, "
                "even trivial values. Chain tools when needed."
            ),
            mcp_servers=[math_mcp, text_mcp],
        )

        questions = [
            # Test 1 — math only
            "What is 13 times 29, then raised to the power of 2?",
            # Test 2 — text only
            "Reverse the string 'model context protocol' and then shout it.",
            # Test 3 — crosses both servers
            "Count the words in the sentence 'MCP makes tools discoverable at runtime' "
            "and then multiply that count by 13.5.",
        ]

        for i, q in enumerate(questions, 1):
            print(f"\n────────── Test {i} ──────────")
            print(f"❓ {q}")
            result = await Runner.run(starting_agent=agent, input=q)
            print(f"💬 {result.final_output}")


# Top-level await — the notebook already has a running event loop,
# so `asyncio.run(...)` would raise. This mirrors the earlier cells.
await agents_sdk_demo()

In [7]:
# ══════════════════════════════════════════════════════════════
# OpenAI Agents SDK + MCP over HTTP — connect to pre-started servers                         
# Requires the HTTP bonus servers to be running (math_server_http.py on 8001,              
# text_server_http.py on 8002), either in terminals or via the Popen cell.                   
# ══════════════════════════════════════════════════════════════                             
from agents import Agent, Runner                                                             
from agents.mcp import MCPServerStreamableHttp                                               
                                                                                            
MATH_URL = "http://127.0.0.1:8001/mcp/"                                                      
TEXT_URL = "http://127.0.0.1:8002/mcp/"
                                                                                            
                                                                                            
async def agents_sdk_http_demo():
    # Note: `params={"url": ...}` — no command, no args. The SDK just opens                  
    # a streamable-http session to the URL. Servers are fully independent.                   
    math_mcp = MCPServerStreamableHttp(                                                      
        params={"url": MATH_URL},                                                            
        cache_tools_list=True,                                                               
    )                                                                                        
    text_mcp = MCPServerStreamableHttp(                                                    
        params={"url": TEXT_URL},
        cache_tools_list=True,                                                               
    )
                                                                                            
    # `async with` still manages the CLIENT session lifecycle,                               
    # but the server processes keep running after this block exits.
    async with math_mcp, text_mcp:                                                           
        agent = Agent(                                                                     
            name="MCP Demo Assistant",                                                       
            model=MODEL,                                                                   
            instructions=(                                                                   
                "You have access to math and text tools over MCP. "
                "Use them for every operation — do not compute anything yourself, "          
                "even trivial values. Chain tools when needed."                            
            ),                                                                               
            mcp_servers=[math_mcp, text_mcp],
        )                                                                                    
                                                                                            
        questions = [
            "What is 13 times 29, then raised to the power of 2?",
            "Reverse the string 'model context protocol' and then shout it.",                
            "Count the words in the sentence 'MCP makes tools discoverable at runtime' "
            "and then multiply that count by 13.5.",                                         
        ]                                                                                    

        for i, q in enumerate(questions, 1):                                                 
            print(f"\n────────── Test {i} ──────────")                                     
            print(f"❓ {q}")
            result = await Runner.run(starting_agent=agent, input=q)                         
            print(f"💬 {result.final_output}")
                                                                                            
                                                                                            
await agents_sdk_http_demo()


────────── Test 1 ──────────
❓ What is 13 times 29, then raised to the power of 2?
💬 13 × 29 = 377, and 377^2 = 142129.

────────── Test 2 ──────────
❓ Reverse the string 'model context protocol' and then shout it.
💬 LOCOTORP TXETNOC LEDOM

────────── Test 3 ──────────
❓ Count the words in the sentence 'MCP makes tools discoverable at runtime' and then multiply that count by 13.5.
💬 Word count: 6
Product (6 × 13.5): 81.0


---
## Bonus — Prestart servers (HTTP transport)

Everything above has the host **spawn each server as a subprocess** (`StdioServerParameters` + `stdio_client`). Convenient for a demo, but unrealistic:

- Host and server share a process tree — kill the host, the servers die.
- One server can't be shared across multiple hosts.
- Servers can't live on a different machine.
- Restarting the host re-imports and re-initializes every server.

Real deployments run servers as **independent services** — usually over HTTP — and the host just points at a URL. MCP supports this out of the box via the `streamable-http` transport.

What changes:

| Step | stdio version | HTTP version |
|---|---|---|
| Server runs with | `mcp.run()` | `mcp.run(transport="streamable-http")` on a port |
| Started by | Host (subprocess) | Whatever you want — terminal, systemd, Docker, k8s |
| Client connects via | `stdio_client(StdioServerParameters(...))` | `streamablehttp_client("http://host:port/mcp")` |
| Host knows about server via | command + args | **URL** |

Everything else — `ClientSession`, `list_tools`, `call_tool` — is **identical**. That's the beauty of the protocol: swap the transport, keep the rest.

> One-time install if needed: `pip install "mcp[cli]"` — pulls in uvicorn/starlette for the HTTP transport.

In [40]:
%%writefile math_server_http.py
"""Same math tools as before — but served over HTTP instead of stdio."""
from mcp.server.fastmcp import FastMCP

# FastMCP takes host/port for HTTP transports.
mcp = FastMCP("math", host="127.0.0.1", port=8001)

@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

@mcp.tool()
def power(base: float, exponent: float) -> float:
    """Raise base to the given exponent."""
    return base ** exponent

if __name__ == "__main__":
    # Exposes the MCP endpoint at http://127.0.0.1:8001/mcp
    mcp.run(transport="streamable-http")


Writing math_server_http.py


In [41]:
%%writefile text_server_http.py
"""Same text tools as before — over HTTP on a different port."""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("text", host="127.0.0.1", port=8002)

@mcp.tool()
def uppercase(text: str) -> str:
    """Convert text to uppercase."""
    return text.upper()

@mcp.tool()
def word_count(text: str) -> int:
    """Count the number of whitespace-separated words in text."""
    return len(text.split())

@mcp.tool()
def reverse(text: str) -> str:
    """Return the text with its characters reversed."""
    return text[::-1]

if __name__ == "__main__":
    # Exposes the MCP endpoint at http://127.0.0.1:8002/mcp
    mcp.run(transport="streamable-http")


Writing text_server_http.py


### Step A — Start the servers independently

The whole point is that the host no longer launches them. Two ways:

**Option 1 — Real terminals (most realistic).** In this notebook's directory, open two terminals and run:

```bash
# terminal 1
python math_server_http.py
# -> INFO:     Uvicorn running on http://127.0.0.1:8001

# terminal 2
python text_server_http.py
# -> INFO:     Uvicorn running on http://127.0.0.1:8002
```

Leave them running. The notebook cell below will connect via URL.

**Option 2 — `subprocess.Popen` from the notebook.** Convenient for a self-contained demo; less realistic because the servers die when you shut the kernel. The next cell handles this for you — just skip its `subprocess.Popen` block if you already started them in terminals.

> With Option 1 you can `Ctrl-C` a server and restart it while the notebook is running — the server has a lifecycle independent of the host, exactly like a real microservice.

In [1]:
import os, json
from dotenv import load_dotenv

import textwrap

import truststore
truststore.inject_into_ssl()

def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

API key loaded successfully.


In [2]:
import os                                                 
# Clear proxy vars for this process (the VPN set them system-wide).                          
for v in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY",                                          
        "http_proxy", "https_proxy", "all_proxy"):                                         
    os.environ.pop(v, None)                                                                  
# Belt-and-braces: tell anything that still reads env to bypass loopback.                    
os.environ["NO_PROXY"] = "127.0.0.1,localhost,::1"                                           
os.environ["no_proxy"] = "127.0.0.1,localhost,::1" 

In [3]:
import json
from openai import OpenAI

openai_client = OpenAI()
MODEL = "gpt-5-nano"


async def chat(host, user_message, verbose=True):
    """Run one user message through an MCP-backed tool-use loop."""
    input_items = [{"role": "user", "content": user_message}]

    while True:
        response = openai_client.responses.create(
            instructions="You must use the provided tools for all operations if possible. Do not compute anything yourself, even trivial values. Chain tools when needed.",
            model=MODEL,
            input=input_items,
            tools=host.openai_tools,
        )

        # Collect any tool calls the model emitted this turn
        tool_calls = [item for item in response.output if item.type == "function_call"]

        if not tool_calls:
            # No more tool calls — the model is done; return its text reply
            print("No more tool calls — the model is done; returning its text reply.")
            return response.output_text

        # For each tool call: execute via the host and append result
        for tc in tool_calls:
            args = json.loads(tc.arguments)
            if verbose:
                print(f"🔧 {tc.name}({args})")

            result = await host.call(tc.name, args)

            if verbose:
                print(f"   → {result}")

            # Echo the function_call back, then supply its output
            input_items.append({
                "type": "function_call",
                "call_id": tc.call_id,
                "name": tc.name,
                "arguments": tc.arguments,
            })
            input_items.append({
                "type": "function_call_output",
                "call_id": tc.call_id,
                "output": result,
            })


In [ ]:
# Self-contained variant: Popen the HTTP servers in the background,
# connect by URL, run the same three tests, then clean up.
# If you started servers in separate terminals (Option 1), skip the Popen
# block and jump straight to building MCPHttpHost.

import subprocess, time
from mcp.client.streamable_http import streamable_http_client
from contextlib import AsyncExitStack
from mcp import ClientSession

MATH_URL = "http://127.0.0.1:8001/mcp/"
TEXT_URL = "http://127.0.0.1:8002/mcp/"


time.sleep(2)  # give uvicorn a moment to bind to the ports
print(f"Servers running on {MATH_URL} and {TEXT_URL}")


# --- 2) An HTTP-aware host ------------------------------------------------
# Same public interface as MCPHost (add_server / call / close) so our
# existing chat() loop works UNCHANGED. Only the transport differs.

class MCPHttpHost:
    def __init__(self):
        self.session_by_tool = {}
        self.openai_tools = []
        self._stack = AsyncExitStack()

    async def add_server(self, url):
        # streamablehttp_client yields (read, write, get_session_id)
        read, write, _ = await self._stack.enter_async_context(
            streamable_http_client(url)
        )
        session = await self._stack.enter_async_context(ClientSession(read, write))
        await session.initialize()

        listed = await session.list_tools()
        for t in listed.tools:
            self.session_by_tool[t.name] = session
            self.openai_tools.append({
                "type": "function",
                "name": t.name,
                "description": t.description or "",
                "parameters": t.inputSchema,
            })
        print(f"  connected to {url} — tools: {[t.name for t in listed.tools]}")

    async def call(self, tool_name, args):
        session = self.session_by_tool[tool_name]
        result = await session.call_tool(tool_name, args)
        return "\n".join(c.text for c in result.content if hasattr(c, "text"))

    async def close(self):
        await self._stack.aclose()


# --- 3) Run the same three tests, now over HTTP ---------------------------
http_host = MCPHttpHost()
try:
    print("\nConnecting via HTTP…")
    await http_host.add_server(MATH_URL)
    await http_host.add_server(TEXT_URL)

    for q in [
        "What is 13 times 29, then raised to the power of 2?",
        "Reverse the string 'model context protocol' and then shout it.",
        "Count the words in the sentence 'MCP makes tools discoverable at runtime' "
        "and then multiply that count by 13.5.",
    ]:
        print(f"\n❓ {q}")
        answer = await chat(http_host, q)
        print(f"💬 {answer}")
finally:
    # --- 4) Tear down -----------------------------------------------------
    await http_host.close()
    print("\nServers stopped, clients closed.")

Servers running on http://127.0.0.1:8001/mcp/ and http://127.0.0.1:8002/mcp/

Connecting via HTTP…
  connected to http://127.0.0.1:8001/mcp/ — tools: ['add', 'multiply', 'power']
  connected to http://127.0.0.1:8002/mcp/ — tools: ['uppercase', 'word_count', 'reverse']

❓ What is 13 times 29, then raised to the power of 2?
🔧 multiply({'a': 13, 'b': 29})
   → 377.0
🔧 power({'base': 377, 'exponent': 2})
   → 142129.0
No more tool calls — the model is done; returning its text reply.
💬 142129.0

❓ Reverse the string 'model context protocol' and then shout it.
🔧 reverse({'text': 'model context protocol'})
   → locotorp txetnoc ledom
🔧 uppercase({'text': 'locotorp txetnoc ledom'})
   → LOCOTORP TXETNOC LEDOM
No more tool calls — the model is done; returning its text reply.
💬 LOCOTORP TXETNOC LEDOM

❓ Count the words in the sentence 'MCP makes tools discoverable at runtime' and then multiply that count by 13.5.
🔧 word_count({'text': 'MCP makes tools discoverable at runtime'})
   → 6
🔧 multiply

---
## Bonus — Consuming a third-party MCP server (`mcp-server-git`)

So far every server was **ours**. The other half of MCP's promise is plugging in **servers someone else wrote** — no integration code, no custom schemas, no API-key dance. Our existing client just spawns their binary and talks protocol.

We'll use the official Git server: `mcp-server-git`. It exposes tools like `git_status`, `git_log`, `git_diff`, `git_show` — all scoped to a repo path we give it. No auth needed: it just shells out to `git` locally.

```bash
pip install mcp-server-git   # one-time
```

**One caveat about "starting the server yourself":** `mcp-server-git` is **stdio-only** — it reads JSON-RPC from stdin and writes to stdout. That means a process running in a terminal can't be talked to by another process; stdio requires the parent to spawn the child. So there are two realistic options:

| Option | You do | Notebook does |
|---|---|---|
| **A. You own the repo, notebook spawns the server** | clone / pick a real repo | spawn `python -m mcp_server_git` as a stdio subprocess pointing at your repo |
| **B. You own the repo AND the server process** (via `mcp-proxy`) | `mcp-proxy --sse-port 8765 -- python -m mcp_server_git --repository <path>` | connect via SSE to `http://127.0.0.1:8765/sse` |

Option A is what we'll do by default. Option B is at the bottom if you truly want the server as a standalone process.

In [ ]:
# --- Step 1: clone a real online repo (or point at one you already have) --
# Pick any small public repo. octocat/Hello-World is tiny and canonical.
# If you've already cloned something elsewhere, just set REPO_PATH to it.

import subprocess, pathlib

REPO_URL  = "https://github.com/octocat/Hello-World.git"
REPO_PATH = pathlib.Path("/tmp/mcp_external_repo")

if not (REPO_PATH / ".git").exists():
    print(f"Cloning {REPO_URL} → {REPO_PATH} …")
    REPO_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--quiet", REPO_URL, str(REPO_PATH)],
        check=True,
    )
else:
    print(f"Using existing clone at {REPO_PATH}")

# Confirm we're looking at a real repo with real commits.
log_preview = subprocess.run(
    ["git", "log", "--oneline", "-n", "3"],
    cwd=REPO_PATH, check=True, capture_output=True, text=True,
).stdout
print(f"\nMost recent commits:\n{log_preview}")

In [ ]:
# --- Step 2: spawn mcp-server-git and talk protocol -----------------------
# Same client pattern as Step 3 of this notebook. Only the
# StdioServerParameters change — our code has no idea it's "git".
#
# Note: every mcp-server-git tool takes `repo_path` as a required argument
# in addition to the `--repository` CLI flag. Quirk of that server.

git_params = StdioServerParameters(
    command=sys.executable,
    args=["-m", "mcp_server_git", "--repository", str(REPO_PATH)],
)

async with stdio_client(git_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        # 1) DISCOVERY — what does this server expose?
        listed = await session.list_tools()
        print(f"mcp-server-git exposes {len(listed.tools)} tools:\n")
        for t in listed.tools:
            first_line = (t.description or "").split("\n")[0][:70]
            print(f"  • {t.name:22} — {first_line}")

        def show(label, result, limit=600):
            text = "\n".join(c.text for c in result.content if hasattr(c, "text"))
            print(f"\n── {label} ──")
            print(text[:limit] + ("…" if len(text) > limit else ""))

        # 2) git_status — working tree state
        show("git_status",
             await session.call_tool("git_status", {"repo_path": str(REPO_PATH)}))

        # 3) git_log — last 5 commits from a real public repo
        show("git_log (last 5)",
             await session.call_tool("git_log",
                                     {"repo_path": str(REPO_PATH), "max_count": 5}))

        # 4) git_show — full details of HEAD (commit message + diff)
        show("git_show HEAD",
             await session.call_tool("git_show",
                                     {"repo_path": str(REPO_PATH), "revision": "HEAD"}))

---
### Option B (optional) — You start the server yourself in a terminal

If you really want `mcp-server-git` to live as an independent process, you need to put HTTP on the outside of it. The common way is **`mcp-proxy`** — a tiny utility that wraps any stdio MCP server and re-exposes it over SSE. The stdio server doesn't know anything changed; your client just talks HTTP/SSE to the proxy.

**Install once:**
```bash
pip install mcp-proxy
```

**Start it yourself in a terminal** (leave it running):
```bash
mcp-proxy --sse-port 8765 \
  -- python -m mcp_server_git --repository /tmp/mcp_external_repo
# -> INFO:  Uvicorn running on http://127.0.0.1:8765
```

Everything after `--` is the stdio command that `mcp-proxy` will spawn internally and relay for. You can `Ctrl-C` it, change repos, restart — the notebook reconnects on the next call.

**Then connect from the notebook via SSE** (not `streamablehttp_client` — `mcp-proxy` speaks the older SSE transport by default):

```python
from mcp.client.sse import sse_client

SSE_URL = "http://127.0.0.1:8765/sse"

async with sse_client(SSE_URL) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        listed = await session.list_tools()
        print("Tools via mcp-proxy:", [t.name for t in listed.tools])
        result = await session.call_tool(
            "git_log",
            {"repo_path": str(REPO_PATH), "max_count": 3},
        )
        print(result.content[0].text[:400])
```

Conceptually you now have the same architecture as the HTTP transport bonus above, but for a third-party stdio-only server. That's the pattern you'll use in real deployments whenever you want to host a stdio server behind a URL.

> VPN / proxy env note: if `streamable_http` hung for you earlier, SSE will hang for the same reason. Keep the `NO_PROXY` / proxy-unset snippet from that cell in the kernel.